# Defaqto Sales DB — 04 · Row access policies

**Author:** Ketki Kothe · Solution Engineer, Snowflake · `Snowflake Solution Engineering`
**Workshop:** Snowflake × Defaqto, Tuesday 8 September 2026, Snowflake London

Notebook 01 built the gold tables. This one puts **row-level security** on them, so one
insurer sees one insurer — and proves it.

**Why this is the commercial notebook.** Ben wants to sell partner reporting as a premium
service. That only works if a provider can be given a login and see *only themselves*,
with no possibility of seeing a competitor. That guarantee cannot live in application
code, because the next person to write an app — or anyone with a worksheet — would bypass
it. It has to live on the table.

**You will do all of this yourself**, not watch it. By the end you will have:

- a mapping table saying which role sees which insurer
- a row access policy on three gold tables
- a role for one insurer
- a **user** for that insurer, with exactly one role
- proof that the user sees one insurer and you see all seven

Run notebook 01 first. This notebook reads and alters the gold tables it creates.

## Pick your insurer — do this before you run anything

Roles and users are **account-level** objects, not schema-level. Everything else in this
notebook lives in your own schema and cannot clash with anyone else's, but two people
creating `PARTNER_VEYGO` will collide.

So agree who takes what:

| Person | Insurer | Role | User |
| --- | --- | --- | --- |
| Ketki | `ZIXTY` | `PARTNER_ZIXTY` | `ZIXTY_USER` |
| Mike | `VEYGO` | `PARTNER_VEYGO` | `VEYGO_USER` |
| a second attendee | `COVERTIME` | `PARTNER_COVERTIME` | `COVERTIME_USER` |

Spare insurers if you need a fourth or fifth: `SAFELYINSURED`, `STERLING`, `ZIXTY PLUS`.

Comparison sites are deliberately out of scope here. The affiliate IDs do not reconcile
between the quote and sales systems — the sales feed books some traffic under an
affiliate that never appears in the quote feed — so partner attribution by comparison
site is not yet safe to publish. That is a real finding, not a gap in the notebook.

In [ ]:
%%sql -r dataframe_1
-- ===========================================================================
-- EDIT THESE THREE, then run every cell in order.
-- ===========================================================================
SET my_alias   = 'kkothe';        -- same alias you used in notebook 01
SET my_insurer = 'ZIXTY';         -- ZIXTY | VEYGO | COVERTIME | SAFELYINSURED
SET my_password = 'ChangeMe_2026!';   -- for the new partner user. Yours to choose.

-- Everything below is derived, so you never type a name twice.
SET my_schema = (SELECT 'DEFAQTO_DB.TRANSFORMED_' || UPPER($my_alias));
SET my_role   = (SELECT 'PARTNER_' || REPLACE(UPPER($my_insurer), ' ', '_'));
SET my_user   = (SELECT REPLACE(UPPER($my_insurer), ' ', '_') || '_USER');
SET me        = (SELECT '"' || CURRENT_USER() || '"');

USE SCHEMA IDENTIFIER($my_schema);

SELECT $my_schema AS building_in, $my_insurer AS insurer,
       $my_role AS role_to_create, $my_user AS user_to_create, $me AS you;

## 1 · The mapping table

The policy needs to know which role is allowed to see which insurer. You could hard-code
that in the policy body, but then adding a partner means altering a security object.

A mapping table means **adding a partner is an `INSERT`**. That is the difference between
onboarding a customer in seconds and raising a change request.

In [ ]:
%%sql -r dataframe_4
CREATE TABLE IF NOT EXISTS PARTNER_ACCESS (
    ROLE_NAME     VARCHAR NOT NULL,
    PARTNER_TYPE  VARCHAR NOT NULL,   -- 'PROVIDER' here; 'PCW' is out of scope
    PROVIDER_KEY  VARCHAR,
    AFFILIATE_ID  NUMBER
);

-- MERGE rather than INSERT so re-running this notebook does not duplicate your row.
MERGE INTO PARTNER_ACCESS t
USING (SELECT $my_role AS ROLE_NAME, 'PROVIDER' AS PARTNER_TYPE,
              $my_insurer AS PROVIDER_KEY, NULL AS AFFILIATE_ID) s
   ON t.ROLE_NAME = s.ROLE_NAME
WHEN NOT MATCHED THEN INSERT (ROLE_NAME, PARTNER_TYPE, PROVIDER_KEY, AFFILIATE_ID)
     VALUES (s.ROLE_NAME, s.PARTNER_TYPE, s.PROVIDER_KEY, s.AFFILIATE_ID);

SELECT * FROM PARTNER_ACCESS ORDER BY ROLE_NAME;

## 2 · The row access policy

**The lookup** — joins `CURRENT_ROLE()` to the mapping table and compares the mapped
insurer to the row's `PROVIDER_KEY`. The `provider_key` in lowercase is the *policy
argument*, i.e. the value of the column on the row being tested. The uppercase
`a.PROVIDER_KEY` is the mapping table's column. They are different things and the
lowercase one is easy to miss.



In [ ]:
%%sql -r dataframe_5
CREATE ROW ACCESS POLICY IF NOT EXISTS PROVIDER_RAP
AS (PROVIDER_KEY VARCHAR) RETURNS BOOLEAN ->
    -- Escape hatch: internal roles see everything.
    CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN')
    -- Otherwise: only the rows your role is mapped to.
    OR EXISTS (
        SELECT 1
        FROM PARTNER_ACCESS a
        WHERE a.ROLE_NAME    = CURRENT_ROLE()
          AND a.PARTNER_TYPE = 'PROVIDER'
          AND a.PROVIDER_KEY = PROVIDER_KEY
    )
COMMENT = 'One insurer sees one insurer. Internal roles see all.';

SHOW ROW ACCESS POLICIES LIKE 'PROVIDER_RAP';

## 3 · Attach it

**These are dynamic tables and the policy works on them exactly as it would on a normal
table.** That is worth noticing: the security travels with the table, not with the
pipeline that maintains it.

Two errors you will hit if you skip the `DROP ALL`:

- `ALTER ... ADD ROW ACCESS POLICY` → *"Object X already has a ROW_ACCESS_POLICY. Only one
  ROW_ACCESS_POLICY is allowed at a time."*
- `CREATE OR REPLACE ROW ACCESS POLICY` → *"cannot be dropped/replaced as it is associated
  with one or more entities."*

`DROP ALL ROW ACCESS POLICIES` is safe when nothing is attached, so the pair below is
re-runnable. You will want that while you iterate.

In [ ]:
%%sql -r dataframe_6

ALTER DYNAMIC TABLE GOLD_PROVIDER_DAILY    DROP ALL ROW ACCESS POLICIES;
ALTER DYNAMIC TABLE GOLD_COHORT_CONVERSION DROP ALL ROW ACCESS POLICIES;

ALTER DYNAMIC TABLE GOLD_PROVIDER_DAILY    ADD ROW ACCESS POLICY PROVIDER_RAP ON (PROVIDER_KEY);
ALTER DYNAMIC TABLE GOLD_COHORT_CONVERSION ADD ROW ACCESS POLICY PROVIDER_RAP ON (PROVIDER_KEY);

In [ ]:
%%sql -r dataframe_7
-- Never assume an attach worked. This is the authoritative check.
SELECT 'GOLD_PROVIDER_DAILY' AS table_name, POLICY_NAME
FROM TABLE(INFORMATION_SCHEMA.POLICY_REFERENCES(
    REF_ENTITY_NAME   => $my_schema || '.GOLD_PROVIDER_DAILY',
    REF_ENTITY_DOMAIN => 'TABLE'))
UNION ALL
SELECT 'GOLD_COHORT_CONVERSION', POLICY_NAME
FROM TABLE(INFORMATION_SCHEMA.POLICY_REFERENCES(
    REF_ENTITY_NAME   => $my_schema || '.GOLD_COHORT_CONVERSION',
    REF_ENTITY_DOMAIN => 'TABLE'));

## 4 · The role

One role per insurer, holding the least it can. Note what is **not** here: no privileges
on the silver or raw layers, and no `CREATE` anything. A partner can read three gold
tables and nothing else.

In [ ]:
%%sql -r dataframe_8
CREATE ROLE IF NOT EXISTS IDENTIFIER($my_role)
  COMMENT = 'Sees only one insurer. Row access policy does the filtering.';

-- IDENTIFIER() accepts a plain variable but NOT a concatenated expression in a
-- GRANT, so build the fully qualified names up front.
SET t_provider = $my_schema || '.GOLD_PROVIDER_DAILY';
SET t_cohort   = $my_schema || '.GOLD_COHORT_CONVERSION';
SET t_funnel   = $my_schema || '.GOLD_FUNNEL_DAILY';
SET t_agent    = $my_schema || '.DEFAQTO_ANALYST';
SET t_view     = $my_schema || '.DEFAQTO_INSIGHTS';

GRANT USAGE ON DATABASE  DEFAQTO_DB            TO ROLE IDENTIFIER($my_role);
GRANT USAGE ON SCHEMA    IDENTIFIER($my_schema) TO ROLE IDENTIFIER($my_role);
GRANT USAGE ON WAREHOUSE COMPUTE_WH            TO ROLE IDENTIFIER($my_role);

GRANT SELECT ON DYNAMIC TABLE IDENTIFIER($t_provider) TO ROLE IDENTIFIER($my_role);
GRANT SELECT ON DYNAMIC TABLE IDENTIFIER($t_cohort)   TO ROLE IDENTIFIER($my_role);
GRANT SELECT ON DYNAMIC TABLE IDENTIFIER($t_funnel)   TO ROLE IDENTIFIER($my_role);

-- Needed for the 14:30 demo: the partner must be able to ask the agent.
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER       TO ROLE IDENTIFIER($my_role);
GRANT USAGE      ON AGENT         IDENTIFIER($t_agent) TO ROLE IDENTIFIER($my_role);
GRANT REFERENCES ON SEMANTIC VIEW IDENTIFIER($t_view)  TO ROLE IDENTIFIER($my_role);
GRANT SELECT     ON SEMANTIC VIEW IDENTIFIER($t_view)  TO ROLE IDENTIFIER($my_role);

-- Grant the role to YOURSELF too. This is what lets you verify with USE ROLE in the next
-- cell instead of logging out, which would cost ten minutes before you knew if it worked.
GRANT ROLE IDENTIFIER($my_role) TO USER IDENTIFIER($me);

SHOW GRANTS TO ROLE IDENTIFIER($my_role);

## 5 · Prove it — the payoff

Same table, same query, two roles. Run both cells and compare.

In [ ]:
USE ROLE IDENTIFIER($my_role);

SELECT CURRENT_ROLE()                  AS acting_as,
       COUNT(DISTINCT PROVIDER_KEY)    AS insurers_visible,
       COUNT(*)                        AS rows_visible
FROM IDENTIFIER($t_provider);  

In [ ]:
%%sql -r dataframe_10
USE ROLE ACCOUNTADMIN;

SELECT CURRENT_ROLE()                  AS acting_as,
       COUNT(DISTINCT PROVIDER_KEY)    AS insurers_visible,
       COUNT(*)                        AS rows_visible
FROM IDENTIFIER($t_provider);  

One insurer against seven. Nothing in the query changed — no `WHERE` clause, no view, no
application logic. **The table itself behaves differently depending on who is asking.**

That is the sentence to remember: a filter in a dashboard protects one dashboard, a row
access policy protects the data.

## 6 · The user



In [ ]:
%%sql -r dataframe_11
CREATE USER IF NOT EXISTS IDENTIFIER($my_user)
  PASSWORD                = $my_password
  MUST_CHANGE_PASSWORD    = FALSE
  DEFAULT_ROLE            = $my_role
  DEFAULT_WAREHOUSE       = 'COMPUTE_WH'
  DEFAULT_SECONDARY_ROLES = ()
  COMMENT                 = 'Partner login. Sees one insurer only.';

-- IF NOT EXISTS skips everything when re-running, so set the properties explicitly too.
ALTER USER IDENTIFIER($my_user) SET
  DEFAULT_ROLE            = $my_role,
  DEFAULT_WAREHOUSE       = 'COMPUTE_WH',
  DEFAULT_SECONDARY_ROLES = ();

GRANT ROLE IDENTIFIER($my_role) TO USER IDENTIFIER($my_user);

In [ ]:
%%sql -r dataframe_12
-- Check what the user RESOLVED to, not what you think you set. A blank default
-- warehouse here is the single most common reason a partner's agent chat fails, and
-- the error it produces never mentions Cortex.
DESCRIBE USER IDENTIFIER($my_user);

SELECT "property" AS setting, "value" AS resolved
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "property" IN ('NAME', 'DEFAULT_ROLE', 'DEFAULT_WAREHOUSE',
                     'DEFAULT_SECONDARY_ROLES', 'DISABLED', 'HAS_PASSWORD');

## 7 · Log in as your partner

1. Open a **private / incognito** window — otherwise you will log yourself out.
2. Sign in with the user and password you set above.
3. Run this, and notice you cannot even see the other insurers exist:

```sql
SELECT PROVIDER_KEY, SUM(SALES) AS sales, SUM(GWP) AS gwp
FROM DEFAQTO_DB.TRANSFORMED_<the alias whose schema you were granted>.GOLD_PROVIDER_DAILY
GROUP BY PROVIDER_KEY;
```

4. Then ask the **agent** something — *"how many policies did I sell?"* The answer is
   filtered too, because agents run as the user's default role. That is the line worth
   having ready: *even the AI can only see your own data, and not because we told it not
   to look.*

**What you will NOT see is the app filtered**, if you open `DEFAQTO_..._SIMPLE`. Those run
with **owner's rights**, so the policy is evaluated against whoever owns the app rather
than whoever is viewing it. `DEFAQTO_PARTNER_INSIGHTS` uses restricted caller's rights and
does filter per viewer. Two apps, same tables, different answers — and the difference is
the runtime, not the policy.